In [33]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

In [34]:
# 데이터 확인하기 2025.11.21
# 이상인 컬럼 제거 후 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np
from scipy.special import logit
import time
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler # 데이터 전처리용
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, recall_score, precision_score
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing

# 모듈 reload
importlib.reload(preprocessing)
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns
from utils.user_utils    import get_model_train_eval
from utils.model_utils   import save_model, load_model
from utils.evaluation import evaluate_model_cv

# train = pd.read_csv("../data/train.csv")
# test  = pd.read_csv("../data/test.csv")

In [35]:
# ============================================================
# 1. 데이터 로드 + ID / TARGET 분리
# ============================================================
print("="*100)
print("Step1. 데이터 로드 및 ID / TARGET 분리")
print("="*100)

DATA_DIR = os.path.join(project_root, "data")
DOC_DIR  = os.path.join(project_root, "doc")

train_path = os.path.join(DATA_DIR, "train.csv")
test_path  = os.path.join(DATA_DIR, "test.csv")

train_raw = pd.read_csv(train_path)
test_raw  = pd.read_csv(test_path)

print(f"train_raw shape: {train_raw.shape}")
print(f"test_raw  shape: {test_raw.shape}")
print(train_raw.head(3))

ID_COL = "ID"
TARGET_COL = "TARGET"

y = train_raw[TARGET_COL].copy()
X = train_raw.drop(columns=[ID_COL, TARGET_COL]).copy()
X_test = test_raw.drop(columns=[ID_COL]).copy()

print("\nID / TARGET 제거 후")
print(f"X shape      : {X.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y shape      : {y.shape}")

Step1. 데이터 로드 및 ID / TARGET 분리
train_raw shape: (76020, 371)
test_raw  shape: (75818, 370)
   ID  var3  var15  imp_ent_var16_ult1  imp_op_var39_comer_ult1  \
0   1     2     23                 0.0                      0.0   
1   3     2     34                 0.0                      0.0   
2   4     2     23                 0.0                      0.0   

   imp_op_var39_comer_ult3  imp_op_var40_comer_ult1  imp_op_var40_comer_ult3  \
0                      0.0                      0.0                      0.0   
1                      0.0                      0.0                      0.0   
2                      0.0                      0.0                      0.0   

   imp_op_var40_efect_ult1  imp_op_var40_efect_ult3  ...  \
0                      0.0                      0.0  ...   
1                      0.0                      0.0  ...   
2                      0.0                      0.0  ...   

   saldo_medio_var33_hace2  saldo_medio_var33_hace3  saldo_medio_var33_ult1  \

In [37]:
# ============================================================
# 2. Zero Count 기반 컬럼 제거 (파일만 불러오기)
#    - doc/remove_cols_train_0.99.txt
# ============================================================
print("\n" + "="*100)
print("Step2. Zero Count 기반 컬럼 제거 (팀 파일 사용)")
print("="*100)

zero_file = os.path.join(DOC_DIR, "remove_cols_0.99.txt")
remove_zero_cols = []

with open(zero_file, "r", encoding="utf-8") as f:
    for line in f:
        col = line.strip()
        if col:  # 빈 줄 제외
            remove_zero_cols.append(col)

print(f"파일에서 읽은 제거 대상 컬럼 수: {len(remove_zero_cols)}")

# 실제 존재하는 컬럼만 적용
remove_zero_cols_in_X      = [c for c in remove_zero_cols if c in X.columns]
remove_zero_cols_not_in_X  = [c for c in remove_zero_cols if c not in X.columns]

print(f"Train에 실제 존재하는 제거 컬럼 수: {len(remove_zero_cols_in_X)}")
if remove_zero_cols_not_in_X:
    print("이미 사라진(존재하지 않는) 컬럼 예시:", remove_zero_cols_not_in_X[:10])

X1      = X.drop(columns=remove_zero_cols_in_X)
X_test1 = X_test.drop(columns=remove_zero_cols_in_X, errors="ignore")

print("\nZero Count 제거 후 결과 요약")
print("="*100)
print(f"Train: {X.shape} → {X1.shape}")
print(f"Test : {X_test.shape} → {X_test1.shape}")
print("="*100)

X_current      = X1.copy()
X_test_current = X_test1.copy()


Step2. Zero Count 기반 컬럼 제거 (팀 파일 사용)
파일에서 읽은 제거 대상 컬럼 수: 217
Train에 실제 존재하는 제거 컬럼 수: 217

Zero Count 제거 후 결과 요약
Train: (76020, 369) → (76020, 152)
Test : (75818, 369) → (75818, 152)


In [38]:
# ============================================================
# 3. 상관계수 0.95 이상 컬럼 제거 (파일만 불러오기)
#    - doc/remove_train_0.95.txt
# ============================================================
print("\n" + "="*100)
print("Step3. 상관계수 0.95 이상 컬럼 제거 (팀 파일 사용)")
print("="*100)

corr_file = os.path.join(DOC_DIR, "remove_cols_train_0.95.txt")
remove_corr_cols = []

with open(corr_file, "r", encoding="utf-8") as f:
    for line in f:
        col = line.strip()
        if col:
            remove_corr_cols.append(col)

print(f"파일에서 읽은 상관계수 제거 컬럼 수: {len(remove_corr_cols)}")

remove_corr_cols_in_X      = [c for c in remove_corr_cols if c in X_current.columns]
remove_corr_cols_not_in_X  = [c for c in remove_corr_cols if c not in X_current.columns]

print(f"Train에 실제 존재하는 제거 컬럼 수: {len(remove_corr_cols_in_X)}")
if remove_corr_cols_not_in_X:
    print("이미 사라진(존재하지 않는) 컬럼 예시:", remove_corr_cols_not_in_X[:10])

X2      = X_current.drop(columns=remove_corr_cols_in_X)
X_test2 = X_test_current.drop(columns=remove_corr_cols_in_X, errors="ignore")

print("\n상관계수 제거 후 결과 요약")
print("="*100)
print(f"Train: {X_current.shape} → {X2.shape}")
print(f"Test : {X_test_current.shape} → {X_test2.shape}")
print("삭제된 컬럼 개수:", len(remove_corr_cols_in_X))
print("="*100)

X_current      = X2.copy()
X_test_current = X_test2.copy()


Step3. 상관계수 0.95 이상 컬럼 제거 (팀 파일 사용)
파일에서 읽은 상관계수 제거 컬럼 수: 283
Train에 실제 존재하는 제거 컬럼 수: 66
이미 사라진(존재하지 않는) 컬럼 예시: ['imp_op_var40_comer_ult1', 'imp_op_var40_comer_ult3', 'imp_op_var40_efect_ult1', 'imp_op_var40_efect_ult3', 'imp_op_var40_ult1', 'imp_sal_var16_ult1', 'ind_var1', 'ind_var2_0', 'ind_var2', 'ind_var6_0']

상관계수 제거 후 결과 요약
Train: (76020, 152) → (76020, 86)
Test : (75818, 152) → (75818, 86)
삭제된 컬럼 개수: 66


In [39]:
# ============================================================
# 4. Log1p 변환 (팀 Log1pColumns.txt 사용, 존재하는 컬럼만 적용)
# ============================================================
print("\n" + "="*100)
print("Step4. Log1p 변환 (팀 Log1pColumns.txt 사용)")
print("="*100)

log_file = os.path.join(DOC_DIR, "Log1pColumns.txt")
log_cols_raw = []

with open(log_file, "r", encoding="utf-8") as f:
    for line in f:
        col = line.strip()
        if col:
            log_cols_raw.append(col)

print(f"Log1p 파일에서 읽은 전체 컬럼 수: {len(log_cols_raw)}")

log_cols_in_X     = [c for c in log_cols_raw if c in X_current.columns]
log_cols_not_in_X = [c for c in log_cols_raw if c not in X_current.columns]

print(f"실제 적용 가능한 Log1p 컬럼 수: {len(log_cols_in_X)}")
if log_cols_not_in_X:
    print("현재 데이터에 없는 Log1p 대상 컬럼 예시:", log_cols_not_in_X[:10])

# log1p 적용
for col in log_cols_in_X:
    X_current[col]      = np.log1p(X_current[col])
    X_test_current[col] = np.log1p(X_test_current[col])

print("\nLog1p 변환 적용 완료.")
print("현재 Train shape:", X_current.shape)
print("현재 Test  shape:", X_test_current.shape)
print("="*100)


Step4. Log1p 변환 (팀 Log1pColumns.txt 사용)
Log1p 파일에서 읽은 전체 컬럼 수: 72
실제 적용 가능한 Log1p 컬럼 수: 41
현재 데이터에 없는 Log1p 대상 컬럼 예시: ['Log1p_Colums', 'num_var14_0', 'imp_ent_var16_ult1', 'saldo_var26', 'saldo_medio_var12_hace3', 'saldo_medio_var12_hace2', 'num_op_var41_hace3', 'num_trasp_var11_ult1', 'saldo_medio_var13_corto_hace3', 'var21']

Log1p 변환 적용 완료.
현재 Train shape: (76020, 86)
현재 Test  shape: (75818, 86)


In [40]:
# ============================================================
# 5. Train / Valid 분리 + StandardScaler
# ============================================================
print("\n" + "="*100)
print("Step5. Train / Valid 분리 + 스케일링")
print("="*100)

X_train, X_val, y_train, y_val = train_test_split(
    X_current,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val   shape: {X_val.shape}")
print(f"y_train mean(양성 비율): {y_train.mean():.4f}")
print(f"y_val   mean(양성 비율): {y_val.mean():.4f}")

scaler = StandardScaler()
X_train_scaled_np = scaler.fit_transform(X_train)
X_val_scaled_np   = scaler.transform(X_val)
X_test_scaled_np  = scaler.transform(X_test_current)

X_train_scaled = pd.DataFrame(X_train_scaled_np, columns=X_train.columns, index=X_train.index)
X_val_scaled   = pd.DataFrame(X_val_scaled_np,   columns=X_val.columns,   index=X_val.index)
X_test_scaled  = pd.DataFrame(X_test_scaled_np,  columns=X_test_current.columns, index=X_test_current.index)

print("\n스케일링 후")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled   shape:", X_val_scaled.shape)
print("X_test_scaled  shape:", X_test_scaled.shape)
print("="*100)


Step5. Train / Valid 분리 + 스케일링
X_train shape: (60816, 86)
X_val   shape: (15204, 86)
y_train mean(양성 비율): 0.0396
y_val   mean(양성 비율): 0.0396

스케일링 후
X_train_scaled shape: (60816, 86)
X_val_scaled   shape: (15204, 86)
X_test_scaled  shape: (75818, 86)


In [ ]:
# ============================================================
# 6. 개별 모델 학습 및 평가 (RF / XGB / LGBM)
# ============================================================
print("\n" + "="*100)
print("Step6. RF / XGB / LGBM 학습 및 평가")
print("="*100)

def print_full_metrics(model_name, y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_proba)
    cm  = confusion_matrix(y_true, y_pred)
    cr  = classification_report(y_true, y_pred, digits=4)

    print(f"===== {model_name} 성능 (Threshold: {threshold:.2f}) =====")
    print(f"AUC       : {auc:.4f}")
    print(f"정확도    : {acc:.4f}")
    print(f"정밀도    : {pre:.4f}")
    print(f"재현율    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(cr)
    print()


Step6. RF / XGB / LGBM 학습 및 평가


In [42]:
# ---------------------------
# 6-1) RandomForest
# ---------------------------
print("\n---- RandomForest 학습 ----")
rf_start = time.time()

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
)

rf.fit(X_train_scaled, y_train)
rf_proba_val = rf.predict_proba(X_val_scaled)[:, 1]

print_full_metrics("RandomForest", y_val, rf_proba_val, threshold=0.5)
print(f"실행 시간: {time.time() - rf_start:.4f} sec")


---- RandomForest 학습 ----
===== RandomForest 성능 (Threshold: 0.50) =====
AUC       : 0.8275
정확도    : 0.9604
정밀도    : 0.0000
재현율    : 0.0000
F1-score  : 0.0000

Confusion Matrix:
[[14602     0]
 [  602     0]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9604    1.0000    0.9798     14602
           1     0.0000    0.0000    0.0000       602

    accuracy                         0.9604     15204
   macro avg     0.4802    0.5000    0.4899     15204
weighted avg     0.9224    0.9604    0.9410     15204


실행 시간: 5.2626 sec


In [45]:
# ---------------------------
# 6-2) XGBoost
# ---------------------------
print("\n---- XGBoost 학습 ----")
xgb_start = time.time()

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    reg_lambda=1.0,
)

xgb.fit(X_train_scaled, y_train)
xgb_proba_val = xgb.predict_proba(X_val_scaled)[:, 1]

print_full_metrics("XGBoost", y_val, xgb_proba_val, threshold=0.5)
print(f"실행 시간: {time.time() - xgb_start:.4f} sec")


---- XGBoost 학습 ----
===== XGBoost 성능 (Threshold: 0.50) =====
AUC       : 0.8492
정확도    : 0.9606
정밀도    : 0.7143
재현율    : 0.0083
F1-score  : 0.0164

Confusion Matrix:
[[14600     2]
 [  597     5]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9607    0.9999    0.9799     14602
           1     0.7143    0.0083    0.0164       602

    accuracy                         0.9606     15204
   macro avg     0.8375    0.5041    0.4982     15204
weighted avg     0.9510    0.9606    0.9417     15204


실행 시간: 2.1726 sec


In [46]:
# ---------------------------
# 6-3) LightGBM
# ---------------------------
print("\n---- LightGBM 학습 ----")
lgb_start = time.time()

lgb = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

lgb.fit(X_train_scaled, y_train)
lgb_proba_val = lgb.predict_proba(X_val_scaled)[:, 1]

print_full_metrics("LightGBM", y_val, lgb_proba_val, threshold=0.5)
print(f"실행 시간: {time.time() - lgb_start:.4f} sec")

print("="*100)


---- LightGBM 학습 ----
[LightGBM] [Info] Number of positive: 2406, number of negative: 58410
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006512 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7185
[LightGBM] [Info] Number of data points in the train set: 60816, number of used features: 86
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.039562 -> initscore=-3.189521
[LightGBM] [Info] Start training from score -3.189521
===== LightGBM 성능 (Threshold: 0.50) =====
AUC       : 0.8428
정확도    : 0.9603
정밀도    : 0.3333
재현율    : 0.0033
F1-score  : 0.0066

Confusion Matrix:
[[14598     4]
 [  600     2]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9605    0.9997    0.9797     14602
           1     0.3333    0.0033    0.0066       602

    accuracy                         0.9603     15204


In [47]:
# ============================================================
# 7. Stacking (RF + XGB + LGBM → Logistic Regression)
# ============================================================
print("\n" + "="*100)
print("Step7. Stacking (RF + XGB + LGBM → Logistic Regression)")
print("="*100)

stack_start = time.time()

# 메타 입력: Validation에서 각 모델의 예측 확률
stack_X_val = np.vstack([
    rf_proba_val,
    xgb_proba_val,
    lgb_proba_val
]).T   # shape: (n_val, 3)

meta = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

meta.fit(stack_X_val, y_val)

coef = meta.coef_[0]
base_models = ["RandomForest", "XGBoost", "LightGBM"]

print("개별 모델 기여도(로지스틱 회귀 계수):")
for name, c in zip(base_models, coef):
    print(f"{name} 기여도(계수): {c:.4f}")

stack_proba_val = meta.predict_proba(stack_X_val)[:, 1]
stack_pred_val  = (stack_proba_val >= 0.5).astype(int)

stack_auc = roc_auc_score(y_val, stack_proba_val)
stack_f1  = f1_score(y_val, stack_pred_val, zero_division=0)
stack_rec = recall_score(y_val, stack_pred_val, zero_division=0)

print("\nStacking 모델 ROC-AUC: {:.4f}".format(stack_auc))
print("Stacking 모델 F1-Score: {:.4f}".format(stack_f1))
print("Stacking 모델 Recall:   {:.4f}".format(stack_rec))

print("\n=== Stacking 상세 리포트 ===")
print_full_metrics("Stacking(LogReg on [RF, XGB, LGB])", y_val, stack_proba_val, threshold=0.5)

print(f"Stacking 학습 + 평가 시간: {time.time() - stack_start:.4f} sec")
print("="*100)


Step7. Stacking (RF + XGB + LGBM → Logistic Regression)
개별 모델 기여도(로지스틱 회귀 계수):
RandomForest 기여도(계수): 8.3601
XGBoost 기여도(계수): 7.1733
LightGBM 기여도(계수): 3.8090

Stacking 모델 ROC-AUC: 0.8465
Stacking 모델 F1-Score: 0.2557
Stacking 모델 Recall:   0.6262

=== Stacking 상세 리포트 ===
===== Stacking(LogReg on [RF, XGB, LGB]) 성능 (Threshold: 0.50) =====
AUC       : 0.8465
정확도    : 0.8556
정밀도    : 0.1606
재현율    : 0.6262
F1-score  : 0.2557

Confusion Matrix:
[[12632  1970]
 [  225   377]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9825    0.8651    0.9201     14602
           1     0.1606    0.6262    0.2557       602

    accuracy                         0.8556     15204
   macro avg     0.5716    0.7457    0.5879     15204
weighted avg     0.9500    0.8556    0.8938     15204


Stacking 학습 + 평가 시간: 0.0630 sec
